In [ ]:
from hogsboutils import mailer, email_auth
import pandas as pd
import numpy as np
import datetime

In [ ]:
contacts = pd.read_csv(f'https://docs.google.com/spreadsheets/d/{email_auth["contacts_sheet"]}/export?format=csv&gid=0#',
                       skiprows=3, )
contacts = contacts[2:]

In [ ]:
df = pd.read_csv(f'https://docs.google.com/spreadsheets/d/{email_auth["schedule_sheet"]}/export?format=csv&gid=0#',
                       skiprows=3, names=['date', 'a1', 'a2', 'a3', 'a4'])
areas_se = df.iloc[0].to_dict()
areas_en = df.iloc[1].to_dict()
df.head()

In [ ]:
schedule = df[22:-1]
dates = pd.date_range(start="2026-04-04", periods=len(schedule), freq="7D")
schedule['send_date'] = dates

In [ ]:
#schedule = schedule[schedule.send_date > np.datetime64(datetime.datetime.now())]
schedule

In [ ]:
nested = schedule[['a1', 'a2', 'a3', 'a4']].values
flat_list = [item for sublist in nested for item in sublist]

names = [item for item in flat_list if type(item) is str]
names = list(np.unique(names))

In [ ]:
for name in names:
    if '?' in name:
        continue
    if name not in contacts['Cleaning name'].values:
        print("error, contact not found for", name)

In [ ]:
row_number = np.abs(schedule['send_date'] - datetime.datetime.now()).argmin()
this_date = schedule.iloc[row_number]['date']

In [ ]:
schedule.iloc[row_number]

In [ ]:


def weekly_clean_send(df, week):
    cleaners = df[df.date==week][['a1', 'a2', 'a3', 'a4']].iloc[0].to_dict()
    for area, cleaner in cleaners.items():
        if area =='date':
            continue
        cleaner_row = contacts[contacts['Cleaning name'] == cleaner]
        if len(cleaner_row) == 0:
            print(f"FAIL! for {cleaner} not found in contacts sheet")
            continue
        if len(cleaner_row) > 1:
            if '&' not in cleaner:
                print(f"FAIL! Found unexpected multiple matches for {cleaner} {len(cleaner_row)}")
        for mail in cleaner_row['E-post / Email Address']:
            #print(week, cleaner, mail, areas_en[area])
            #cleaning_mail(cleaner, mail, area)
            a = 1

        

for this_date in schedule.date:
    weekly_clean_send(schedule, this_date)

In [ ]:
def cleaning_mail(responsible, recipient_email, area_code):

    area_se = areas_se[area_code].lower().replace('\n', '')
    area_en = areas_en[area_code].lower().replace('\n', '')
    
    msg = f"<English below> \n\n" \
    f"Hej {responsible.capitalize()}! 🧹🧽✨\n\n"\
    f"En vänlig påminnelse att det är din tur att städa {area_se} den har helgen. " \
    "Om du har frågor kan du kontakta Städgruppen på Discord, eller mejla Lotta på lotta_eklund@yahoo.se. Vänligen svara inte på detta mejl.\n\n" \
    "Med vänliga hälsingar\nStädgruppen och styrelsen.\n\n" \
    f"Hi {responsible.capitalize()}! 🧹🧽✨\n\n"\
    f"A friendly reminder that it is your turn to clean the {area_en} this weekend. " \
    "If you have any questions, kindly contact the cleaning group on Discord, or Lotta at lotta_eklund@yahoo.se. Please do not reply to this email.\n\n" \
    "Kind regards\nCleaning group & board.\n\n" \

    mailer(recipient_email, msg)


### Missing from cleaning schedule

In [ ]:
contacts['Cleaning name'] = contacts['Cleaning name'].astype(str)
contacts[contacts['Cleaning name'] == 'nan']['Namn / Name']